# 21장 · 근거 기반 생성 · 검색 결과가 부족할 때 답변 보류하기

근거 기반 생성에서는 모델에 질문과 참고할 글을 함께 보냅니다. 참고할 글에 답이 없을 때 어떤 응답을 해야 하는지도 지시할 수 있습니다.

## 질문과 근거를 서로 다른 입력 칸에 넣기

`from_messages`로 지시와 사용자 메시지를 정한 뒤, `invoke`에 `context`와 `question`을 함께 넘깁니다. 아래 셀은 수업 요일과 시작 시각 질문이 각각 어느 칸에 들어갔는지 출력합니다. 새 커널에는 `%pip install langchain-core`를 실행하세요.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

grounded = ChatPromptTemplate.from_messages([
    ("system", "근거에서 답을 찾으세요. 없으면 확인할 수 없다고 답하세요."),
    ("human", "근거: {context}\n질문: {question}"),
])
ready = grounded.invoke({"context": "수업은 월요일입니다.", "question": "몇 시에 시작하나요?"})
print(ready.to_messages()[-1].content)

근거: 수업은 월요일입니다.
질문: 몇 시에 시작하나요?


모델에 보낼 글에는 요일만 들어 있습니다. 이 셀은 입력을 조립할 뿐이고, 모델이 시간을 추측하는지는 실제 응답에서 확인합니다.

## 응용 실습 · 근거가 없는 질문에 답하기

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널에서 `%pip install pypdf langchain-openai langchain-core chromadb python-dotenv`를 실행하고 커널을 재시작하세요. 작업 폴더의 .env에 OPENAI_API_KEY를 둡니다. 첫 색인 생성과 새 질문 임베딩은 API를 호출합니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))
from portfolio import load_step, save_step, page_id


### 필터를 거친 원문으로 답하거나 보류하기

20장에 저장한 후보가 남아 있으면 질문과 함께 모델에 보내고, 비었으면 호출을 생략합니다. 후보가 있다는 사실만으로 정답을 보장할 수 없으므로 파일 분할 간격과 녹음 지속 여부가 실제로 적혀 있는지도 확인하세요.

In [ ]:
from rag_lab import answer_from_documents
run = load_step(root, "dated")
def grounded_answer(question, rows):
    if not rows:
        return "확인 보류 · 이 날짜와 기종에 맞는 후보가 없습니다."
    return answer_from_documents(question, rows)

reply = grounded_answer(run["question"], run["eligible"])
save_step(root, "grounded", {**run, "evidence_answer": reply})
print(reply)

`rows`가 빈 경우에는 첫 번째 `return`에서 끝납니다. 후보가 있을 때만 `answer_from_documents`가 실행됩니다. D04의 기본 자료에서는 45분과 녹음이 계속된다는 두 사실을 각각 원문에 대조합니다.

### 실습 · 검색을 비웠을 때 멈추는지 보기

질문을 유지하고 후보만 모두 제거하세요. 보류 사유가 나오고 모델에 보내는 줄에 도달하지 않아야 합니다.

In [ ]:
# grounded_answer에 같은 질문과 빈 리스트를 넣으세요.

<details><summary>정답 코드·출력·풀이</summary><pre><code>print(grounded_answer(run[&quot;question&quot;], []))</code></pre><p>제공 자료로 확인할 값</p><pre>확인 보류 · 이 날짜와 기종에 맞는 후보가 없습니다.</pre><p>빈 목록에서 복귀하는 분기에는 모델 호출이 없습니다.</p></details>

## retrieve의 후보에서 답할 수 있는 주장 가려내기

`retrieve`가 찾는 것은 질문과 가까운 문단입니다. `k=3`은 최대 세 후보를 요청할 뿐, 세 후보에 답이 있다는 약속은 아닙니다. 아래에서 `row["text"][100:235]`는 화면에 띄울 일부 글자만 잘라냅니다. 다음 호출의 `candidates`에는 자르기 전의 원문이 남습니다.

“B형 녹음 파일을 컴퓨터에 무선으로 보낼 수 있나요?”에서 기종·대상·전송 방식을 표시합니다. 소스 PDF의 2쪽을 읽어 **케이블 복사**, **무선 전송 지원**, **무선 전송 미지원** 세 주장 중 어떤 것을 원문만으로 확인할 수 있는지 판단하세요. 지원 방식 목록이 모두 실린 문서인지 모르므로, 언급이 없다는 이유로 미지원이라고 단정하기도 어렵습니다.

In [2]:
from rag_lab import open_index, retrieve, answer_from_documents
index = open_index()
q = "2027년 5월 B형에서 녹음 파일을 무선으로 컴퓨터에 보낼 수 있나요?"
candidates = retrieve(index, q, k=3)
for row in candidates:
    print(row["id"], row["text"][100:235])

recorder-b-1.0.pdf#p2 이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다. 이미 저장한
recorder-a-1.1.pdf#p2 이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다. 이미 저장한
recorder-a-1.0.pdf#p2 립니다. 파일이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다.


검색 거리가 짧다는 것은 주제가 가깝다는 뜻입니다. 쪽에 ‘컴퓨터’, ‘파일’, ‘복사’가 함께 나와도 무선 지원을 확인할 수는 없습니다. 답변 지시에는 사실을 뒷받침하는 범위와, 근거가 부족할 때 사용자에게 알려 줄 정보를 함께 적습니다.

`answer_from_documents(q, candidates)`는 후보의 ID와 본문을 프롬프트에 넣어 답 문자열을 돌려주는 교재 함수입니다. 후보를 추가로 검색하지 않습니다. 따라서 무선 전송을 언급한 문장이 없을 때 “지원하지 않는다”와 “이 자료로 확인할 수 없다” 중 어떤 답을 낼 수 있는지는 모델이 받은 원문으로 판단하세요.

In [3]:
print(answer_from_documents(q, candidates))
# 같은 방식으로 근거가 분명한 질문도 확인합니다.
supported = "2027년 5월 B형에서 케이블로 연결한 뒤 어느 폴더를 열까요?"
print(answer_from_documents(supported, retrieve(index, supported, k=3)))

2027년 5월 B형에서 녹음 파일을 무선으로 컴퓨터에 보낼 수 없습니다. 자료에 따르면, B형 1.0판(2027년 4월 1일부터 적용)에서는 녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하여 녹음기에 있는 RECORD 폴더에서 파일을 복사하도록 안내하고 있습니다. 무선 전송에 대한 언급은 없습니다.[recorder-b-1.0.pdf#p2]
2027년 5월 B형에서 케이블로 연결한 뒤에는 녹음기에 있는 RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다[recorder-b-1.0.pdf#p2].


## answer_from_documents의 답을 제공한 근거와 대조하기

`answer_from_documents(질문, 문단 목록)`은 수업용 함수이고 반환값은 답 문자열입니다. 답할 수 없는 질문과 답할 수 있는 질문을 하나씩 보내면 프롬프트가 두 경우를 구별하는지 확인할 수 있습니다. 답에 나온 행동을 실제로 뒷받침하는 문장을 전달한 목록에서 찾으세요.

지원 여부가 불분명하면 확인을 보류하고, 문서가 다루는 범위가 케이블 연결임을 안내할 수 있습니다. 반대로 두 번째 질문에는 RECORD 폴더라는 근거가 있으므로 답해야 합니다. “확인할 수 없습니다”만 모든 질문에 내놓는 도우미는 첫 사례만 우연히 맞힙니다.

## 문자열에서 확인한 사실과 답이 추가한 주장 구별하기

검색에 실패했을 때 가장 위험한 오답 하나를 원문과 나란히 놓으세요. 아래 문장은 오류를 찾도록 만든 가상 답입니다. 판정할 때는 제품명이나 인용 표시가 그럴듯한지가 아니라, ‘무선 전송을 지원하지 않는다’는 주장이 실제 쪽에 있는지를 읽습니다.

In [4]:
from rag_lab import SOURCES, read_pages
manual_page = read_pages(SOURCES / "recorder-b-1.0.pdf")[1]
print("검토할 가상 답: B형은 무선 전송을 지원하지 않습니다.")
print(manual_page["text"][manual_page["text"].index("파일 옮기기"):])

검토할 가상 답: B형은 무선 전송을 지원하지 않습니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다. 이미 저장한 파일은
남습니다. 파일을 컴퓨터에 복사한 뒤 녹음기에서 지워 공간을
확보하세요.
가상 제품 설명서   B형 1.0판
2 / 2쪽


케이블 복사가 된다는 안내는 보입니다. 전송 방식을 빠짐없이 나열한다고 선언한 문장도, 무선 기능이 없다는 문장도 없습니다. 따라서 “무선 지원 여부는 확인하기 어렵고, 문서로 확인되는 케이블 복사는 이렇게 한다”로 수정합니다. 프로그램으로 단어 하나의 존재만 검사해서 이 판단을 대신하지 않습니다.

사용자에게는 보류 이유와 다음 행동이 필요합니다. 제품이 무엇인지 몰라 답할 수 없을 때는 기종을 묻고, 기종은 알지만 설명서에 근거가 없을 때는 확인한 문서와 그 범위를 알려 주세요. 케이블 복사라는 대안을 안내할 때도 그것이 무선 전송 절차인 것처럼 표현하지 않습니다.

## 실습 · 추가 질문과 보류를 구분하기

① “녹음 중 표시등을 숨길 수 있나요?” ② “2027년 5월 B형의 무선 전송 규격은 무엇인가요?” ③ “같은 B형으로 녹음 중에 표시등 설정을 바꿀 수 있나요?”를 직접 보냅니다. ①에는 추가로 알아야 할 조건, ②에는 문서에 빠진 사실, ③에는 답을 뒷받침할 쪽을 적습니다.

In [5]:
test_questions = ["녹음 중 표시등을 숨길 수 있나요?",
                  "2027년 5월 B형의 무선 전송 규격은 무엇인가요?",
                  "2027년 5월 B형은 녹음 중 표시등 설정을 바꿀 수 있나요?"]
# 질문마다 retrieve로 찾은 쪽 ID를 출력하고 같은 문단들을 answer_from_documents에 넘기세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>for question in [&quot;녹음 중 표시등을 숨길 수 있나요?&quot;,
                 &quot;2027년 5월 B형의 무선 전송 규격은 무엇인가요?&quot;,
                 &quot;2027년 5월 B형은 녹음 중 표시등 설정을 바꿀 수 있나요?&quot;]:
    docs = retrieve(index, question, k=4)
    print(question, answer_from_documents(question, docs))
    print([p[&quot;id&quot;] for p in docs])</code></pre><p>①은 기종을 물어보거나 기종별 차이를 조건부로 안내할 수 있습니다. ②는 규격을 찾지 못했다고 밝혀야 합니다. ③은 B형 1.0판 1쪽에 따라 녹음 전 설정해야 한다고 안내합니다. 세 번째까지 보류했다면 관련 쪽이 후보에 있었는지부터 보세요.</p><p>집필 때 보관한 답변 본문입니다. 현재 실행의 후보 ID는 각 답 바로 아래 줄에서 확인하세요.</p><pre>녹음 중 표시등을 숨길 수 있나요? 들소리 A형(1.0판, 1.1판)에서는 녹음 중 표시등을 끄는 설정이 없습니다. 녹음 중에는 앞면의 빨간 표시등이 켜집니다. 화면 밝기를 낮춰도 표시등은 켜져 있습니다. [recorder-a-1.0.pdf#p1], [recorder-a-1.1.pdf#p1]

들소리 B형(1.0판)에서는 녹음 전에 설정에서 표시등을 ‘숨김’으로 바꾸면 녹음을 시작해도 앞면에 빨간불이 들어오지 않습니다. 단, 이미 녹음 중이면 표시등 설정을 바꿀 수 없고, 녹음을 끝낸 뒤 설정에서 ‘숨김’을 선택하고 새 녹음을 시작해야 합니다. [recorder-b-1.0.pdf#p1]

따라서, 녹음 중 표시등을 숨길 수 있는 기종은 B형 1.0판이며, A형 1.0판과 1.1판에서는 불가능합니다.
2027년 5월 B형의 무선 전송 규격은 무엇인가요? 확인 보류: 2027년 5월 B형의 무선 전송 규격에 관한 정보가 자료에 없습니다.
2027년 5월 B형은 녹음 중 표시등 설정을 바꿀 수 있나요? 2027년 5월 B형은 녹음 중 표시등 설정을 바꿀 수 없습니다.  
자료에 따르면, B형 1.0판(적용 기간 2027년 4월 1일부터)에서 &quot;이미 녹음 중이면 표시등 설정을 바꿀 수 없습니다. 녹음을 끝낸 뒤 설정에서 ‘숨김’을 선택하고 새 녹음을 시작하세요.&quot;라고 명시되어 있습니다.[recorder-b-1.0.pdf#p1]</pre></details>

## 독립 실습 · 보류를 잘 쓴 답 고르기

다음 문구는 **오류를 찾도록 만든 세 가지 가상 답변**입니다. B형 2쪽을 읽고 사실로 확인되는 구절에만 밑줄을 그으세요. 그런 다음 문의자가 지금 할 수 있는 행동을 담아 다시 씁니다.

- “무선 전송을 지원하지 않습니다.”
- “케이블로 파일을 복사할 수 있습니다. 무선 전송 지원 여부는 주어진 설명서에서 확인되지 않습니다.”
- “블루투스를 켜고 기기 이름을 선택하세요.”

다른 학생은 같은 설명서로 “컴퓨터에 어떤 폴더가 보이나요?”를 묻습니다. 보류 문구를 그대로 반복해도 되는지 실행해 확인하세요.

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>케이블로 파일을 복사할 수 있다는 두 번째 답이 자료 범위에 맞습니다. 설명서가 특정 방법 한 가지를 안내했다고 해서 모든 다른 방법을 금지한다고 단정할 수 없습니다. 폴더 이름은 B형 1.0판 2쪽에서 RECORD라고 확인할 수 있으므로 보류하지 않고 직접 답합니다.</p></details>

## 다음 수업에 가져갈 것

22장은 `my-course/rag/grounded.json`을 읽습니다. 정답 쪽이 처음 검색에서 빠졌는지, 맞는 쪽이 들어갔는데도 답이 틀렸는지 두 경우를 나눠 보세요.

근거가 있는 질문과 없는 질문을 짝으로 남기세요. 다음 장에서 정답 쪽이 검색되는 문제와, 검색돼도 답할 수 없는 문제를 따로 평가합니다.